# Tome 1, Chapter 4: Arithmetic Functions (Python companion)

Exercises first, solutions at the end. Computer-science threads: **sieves for multiplicative functions** (computing
$\varphi, \mu, \tau, \sigma$ for all $n \le 10^7$ at once), and a Monte Carlo estimate of $\pi$ from **random gcds**.

In [ ]:
import math
import itertools
import random
import numpy as np
import matplotlib.pyplot as plt
# Reproducible teaching examples; these seeds are not for cryptographic keys.
import random
from sympy.core.random import seed as seed_sympy
random.seed(10104)
seed_sympy(20104)


## Before the experiment: Averages have personalities

$\tau(n)$ jumps violently, yet its average follows a smooth curve. Predict which of $720$ and $727$ has more divisors, then explain why averaging suppresses the jumps. In the Monte Carlo experiment distinguish finite-box bias from sampling error.

**How to use this notebook.** Try the exercise cells first; they are intentionally blank. Then run the solutions in order in a fresh kernel. Assertions check exact identities or a stated finite range; plots and random trials supply evidence, not proofs.


## Exercises

**P4.1 (★) Dirichlet convolution.** Represent an arithmetic function by the list of its values on $1..N$ and implement `conv(f, g)` in $O(N\log N)$ (loop over $d$ and its multiples). Check $\mu * \mathbf 1 = \varepsilon$, $\varphi * \mathbf 1 = \operatorname{id}$, $\mathbf 1 * \mathbf 1 = \tau$ for $N = 10^4$.

In [ ]:
# Your code here


**P4.2 (★★) CS application: a multiplicative sieve.** Compute $\varphi(n)$ and $\mu(n)$ for all $n \le 10^7$ with NumPy, by processing each prime $p$ once (for $\varphi$: multiply the multiples of $p$ by $1 - 1/p$; for $\mu$: flip the sign on multiples of $p$, zero on multiples of $p^2$). Time it, and spot-check against SymPy.

In [ ]:
# Your code here


**P4.3 (★★) $6/\pi^2$ three ways.** Compute $C(N)/N^2$ exactly for $N = 10^4$ with the formula of Exercise 4.8(a); estimate it by sampling random pairs with `math.gcd`; and turn the sampling around to estimate $\pi$. How many samples for 3 correct digits?

In [ ]:
# Your code here


**P4.4 (★★) Average orders.** Plot $\frac1x\sum_{n\le x}\tau(n) - \ln x$ and compare with $2\gamma - 1$. Then guess the average order of $\sigma(n)/n$ numerically (answer: $\zeta(2) = \pi^2/6$).

In [ ]:
# Your code here


**P4.5 (★★) Perfect numbers and primitive words.** List the even perfect numbers coming from the Mersenne primes found in Chapter 2's notebook and check their last digits (Exercise 4.5). Then verify Exercise 4.7 by brute force for binary words of length $n \le 16$.

In [ ]:
# Your code here


---
## Solutions

### P4.1

In [ ]:
N = 10**4
def conv(f, g):
    if len(f) != len(g):
        raise ValueError("use lists with the same cutoff")
    cutoff = len(f) - 1
    h = [0] * (cutoff + 1)
    for d in range(1, cutoff + 1):
        if f[d]:
            for m in range(1, cutoff // d + 1):
                h[d * m] += f[d] * g[m]
    return h

from sympy import factorint
one = [0] + [1] * N
ident = list(range(N + 1))
eps = [0, 1] + [0] * (N - 1)

def from_factorisation(rule):   # multiplicative function from its values rule(p, a) on prime powers
    return [0] + [math.prod(rule(p, a) for p, a in factorint(n).items()) for n in range(1, N + 1)]

mu = from_factorisation(lambda p, a: -1 if a == 1 else 0)
phi = from_factorisation(lambda p, a: p**a - p ** (a - 1))
tau = from_factorisation(lambda p, a: a + 1)
assert conv(mu, one) == eps and conv(phi, one) == ident and conv(one, one) == tau
print("mu*1 = eps, phi*1 = id, 1*1 = tau  (N = 10^4)")

### P4.2

In [ ]:
import time
from sympy import totient, mobius
M = 10**7
t = time.perf_counter()
is_p = np.ones(M + 1, bool); is_p[:2] = False
for p in range(2, math.isqrt(M) + 1):
    if is_p[p]:
        is_p[p * p :: p] = False
primes = np.flatnonzero(is_p)
phi_arr = np.arange(M + 1, dtype=np.int64)
mu_arr = np.ones(M + 1, dtype=np.int8)
for p in primes:
    phi_arr[p::p] -= phi_arr[p::p] // p
    mu_arr[p::p] *= -1
    if p * p <= M:
        mu_arr[p * p :: p * p] = 0
print(f"phi and mu up to 10^7 in {time.perf_counter() - t:.1f} s")
for n in random.sample(range(1, M), 200):
    assert phi_arr[n] == totient(n) and mu_arr[n] == mobius(n)
print("spot checks against SymPy pass")

The update `phi -= phi // p` multiplies by $1 - \frac1p$ exactly, because when prime $p$ is processed, `phi[n]` is still divisible by $p$ (it equals $n\prod(1-1/q)$ over smaller primes $q$, and $v_p(n)$ is unchanged).

### P4.3

In [ ]:
N = 10**4
C = sum(int(mu_arr[d]) * (N // d) ** 2 for d in range(1, N + 1))
print(f"exact C(N)/N^2 = {C / N**2:.6f}   6/pi^2 = {6 / math.pi**2:.6f}")

for samples in [10**4, 10**5, 10**6]:
    hits = sum(math.gcd(random.randrange(1, 10**12), random.randrange(1, 10**12)) == 1 for _ in range(samples))
    print(f"{samples:>8} samples: P ~ {hits / samples:.4f},  pi ~ {math.sqrt(6 * samples / hits):.4f}")

The sample proportion has standard error $\sqrt{p(1-p)/n}$. For the transformed estimate $\widehat\pi=\sqrt{6/\widehat p}$, the delta method gives standard error approximately $1.26/\sqrt n$. Thus $10^7$ samples give an error scale of about $0.0004$, not a guarantee of three correct decimals: even a 95% interval has a half-width of about $0.0008$. Random sampling supplies uncertainty bars, not a promise. The finite box for the sampled integers also introduces a small bias.


### P4.4

In [ ]:
X = 10**6
tau_arr = np.zeros(X + 1, dtype=np.int64)
sigma_arr = np.zeros(X + 1, dtype=np.int64)
for d in range(1, X + 1):
    tau_arr[d::d] += 1
    sigma_arr[d::d] += d
xs = np.unique(np.logspace(1, 6, 200).astype(int))
cum = np.cumsum(tau_arr)
plt.semilogx(xs, cum[xs] / xs - np.log(xs), label=r"$\frac{1}{x}\sum \tau(n) - \ln x$")
plt.axhline(2 * 0.5772156649 - 1, c="k", ls="--", label=r"$2\gamma - 1$"); plt.legend(); plt.show()
n = np.arange(1, X + 1)
print("average of sigma(n)/n up to 10^6:", (sigma_arr[1:] / n).mean(), "  pi^2/6 =", math.pi**2 / 6)

### P4.5

In [ ]:
exponents = [2, 3, 5, 7, 13, 17, 19, 31, 61, 89, 107, 127]
perfect = [2 ** (p - 1) * (2**p - 1) for p in exponents]
print("first perfect numbers:", perfect[:5])
print("last digits:", [x % 10 for x in perfect])

def is_primitive(w):
    n = len(w)
    return all(w != w[:d] * (n // d) for d in range(1, n) if n % d == 0)

from sympy import mobius as mob, divisors
for n in range(1, 17):
    brute = sum(is_primitive("".join(w)) for w in itertools.product("01", repeat=n))
    assert brute == sum(mob(d) * 2 ** (n // d) for d in divisors(n))
print("primitive binary words counted correctly for n <= 16")

## One more question before closing

Derive the uncertainty in sqrt(6/p_hat) from the uncertainty in p_hat. Doubling the number of correct digits costs roughly how many additional samples?

Write one sentence identifying what was proved in the chapter, what was checked exactly by the code, and what remains a conjecture or a heuristic.
